# 05 — SDMX Export (Phase 5)

Converts the merged global outputs into SDMX-CSV long format for submission to UN statistical data portals. Produces three streams: hazard, exposure, population.

## Inputs

- Global CSVs produced by notebook 02

## Outputs

`{output_folder}/sdmx/{hazard|exposure|population}/CSV/HAZARD_SDMX_*.csv`
`{output_folder}/sdmx/{hazard|exposure|population}/CSV/HAZARD_SDMX_*.xlsx`

## Execution order

Run **after** notebook 02.


In [ ]:
# ============================================================
# CCRI Hazard Statistics Processing per country administrative unit 2
# SDMX export for hazard, exposure, and population
# Author: Angelly Pugliese, Ph.D.
# Date: April 2026
# ============================================================

In [ ]:
# ============================================================
# Imports
# ============================================================
import sys
from pathlib import Path

# Anchor everything to the project root so config/bounds/lib resolve regardless of CWD.
PROJECT_ROOT = Path.cwd().parent          # notebooks/ -> handoff/
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
BOUNDS_DIR = PROJECT_ROOT / "bounds"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import pandas as pd
import geopandas as geopd
import ee
import os
import shapely
from GEE_functions import GEEUtils 
from GEE_functions import Hazard
from indicator_functions import IndicatorUtils
import datetime as dt
import csv

import numpy as np
pd.set_option('display.max_columns', 500)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# ============================================================
# Authenticate and initialize the Earth Engine library
# Define GEE asset path for UNICEF CCRI data
# ============================================================
ee.Authenticate()
ee.Initialize(project="unicef-ccri")
unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)
indicator_utils = IndicatorUtils()

In [ ]:
prod_date = dt.date.today().isoformat()

In [ ]:
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]

# Now hazard_list contains Hazard objects
for index, hazard in enumerate(hazard_list):
    print(f"{index}. {hazard.name}, {hazard.code}, {hazard.asset}, {hazard.threshold}, {hazard.threshold_operation}")

In [ ]:
output_folder = f"output_CO_NA"
data_base_path = f"{output_folder}/global/CSV"
prod_date = "2026-05-07"
time_period = "2026"
is_draft = True

# Create the folder to store the SDMX
sdmx_hazard_folder_path = f"{output_folder}/sdmx/hazard"
sdmx_exposure_folder_path = f"{output_folder}/sdmx/exposure"
sdmx_population_folder_path = f"{output_folder}/sdmx/population"
os.makedirs(f"{sdmx_hazard_folder_path}/excel", exist_ok=True)
os.makedirs(f"{sdmx_hazard_folder_path}/CSV", exist_ok=True)
os.makedirs(f"{sdmx_exposure_folder_path}/excel", exist_ok=True)
os.makedirs(f"{sdmx_exposure_folder_path}/CSV", exist_ok=True)
os.makedirs(f"{sdmx_population_folder_path}/excel", exist_ok=True)
os.makedirs(f"{sdmx_population_folder_path}/CSV", exist_ok=True)

# create Hazards SDMX structure
hazards_variable = {
    "hazs_mean": { "unit": "NUMBER", "measure": "HAZARD_MEAN", }, 
    "hazs_std": { "unit": "NUMBER", "measure": "HAZARD_STD" },
    "hazs_max": { "unit": "NUMBER", "measure": "HAZARD_MAX" },
    "hazs_min": { "unit": "NUMBER", "measure": "HAZARD_MIN" },
    "hazs_median": { "unit": "NUMBER", "measure": "HAZARD_MEDIAN" },
    "cindex": { "unit": "IDX", "measure": "HAZARD_COUNTRY_INDEX" },
    "cexpi": { "unit": "IDX", "measure": "COUNTRY_EXPOSURE_INDEX" },
    "cexpc": { "unit": "NUMBER", "measure": "COUNTRY_EXPOSURE_CLASS" }
}
sdmx_hazards_structure = {}
dims_hazards = [
    "DATAFLOW", "REF_AREA", "INDICATOR", "MEASURE", "ADMIN_LEVEL", "ISO3_PARENT", 
    "TIME_PERIOD", "OBS_VALUE", "UNIT_MEASURE", "REGION", "STATUS"
] 

# create Exposure SDMX structure
exposure_variables = {
    "aexp": { "unit": "NUMBER", "measure": "ABSOLUTE_EXPOSURE" }, 
    "amexp": { "unit": "NUMBER", "measure": "ABSOLUTE_EXPOSURE" },
    "afexp": { "unit": "NUMBER", "measure": "ABSOLUTE_EXPOSURE" },
    "u18_aexp": { "unit": "NUMBER", "measure": "ABSOLUTE_EXPOSURE" },
    "u18_amexp": { "unit": "NUMBER", "measure": "ABSOLUTE_EXPOSURE" },
    "u18_afexp": { "unit": "NUMBER", "measure": "ABSOLUTE_EXPOSURE" },

    "rexp": { "unit": "NUMBER", "measure": "RELATIVE_EXPOSURE" }, 
    "rmexp": { "unit": "NUMBER", "measure": "RELATIVE_EXPOSURE" },
    "rfexp": { "unit": "NUMBER", "measure": "RELATIVE_EXPOSURE" },
    "u18_rexp": { "unit": "NUMBER", "measure": "RELATIVE_EXPOSURE" },
    "u18_rmexp": { "unit": "NUMBER", "measure": "RELATIVE_EXPOSURE" },
    "u18_rfexp": { "unit": "NUMBER", "measure": "RELATIVE_EXPOSURE" },
}
dims_exposure = [
    "DATAFLOW", "REF_AREA", "INDICATOR", "MEASURE", "SEX", "AGE", "ADMIN_LEVEL", "ISO3_PARENT", 
    "TIME_PERIOD", "OBS_VALUE", "UNIT_MEASURE", "REGION", "STATUS"
]

# create Population SDMX structure
population_variables = {
    "pop_total": { "unit": "PS", "measure": "POP_TOTAL" }, 
    "pop_total_m": { "unit": "PS", "measure": "POP_M" },
    "pop_total_f": { "unit": "PS", "measure": "POP_F" },
    "u18_pop_total": { "unit": "PS", "measure": "U18_POP_TOTAL" },
    "u18_pop_m": { "unit": "PS", "measure": "U18_POP_M" },
    "u18_pop_f": { "unit": "PS", "measure": "U18_POP_F" }
}
dims_population = [
    "DATAFLOW", "REF_AREA", "INDICATOR", "SEX", "AGE", "ADMIN_LEVEL", "ISO3_PARENT", 
    "TIME_PERIOD", "OBS_VALUE", "UNIT_MEASURE", "REGION", "STATUS"
]


population_calculated = False
for index, hazard in enumerate(hazard_list):
    print(f"Processing SDMX files for hazard {index}: {hazard.name} ({hazard.code})")    
    # Create new SDMX structures for the hazard
    sdmx_hazards_structure = {dim: [] for dim in dims_hazards}  
    sdmx_exposure_structure = {dim: [] for dim in dims_exposure}    
    sdmx_population_structure = {dim: [] for dim in dims_population}    

    excel_path = f"{data_base_path}/{hazard.code}_{prod_date}.csv"
    hazard_df = pd.read_csv(excel_path, index_col=None)
    hazard_df = hazard_df.replace(np.nan, None)
    hazard_name = hazard.name
    hazard_code = hazard.code
    hazard_indicator = hazard.indicator
    admin_level = "2"
    draft_text = "_DRAFT" if is_draft else ""

    for df_row in hazard_df.itertuples(index=False):
        adm2_ucode = getattr(df_row, "adm2_ucode")
        iso3 = getattr(df_row, "iso3")
        region = getattr(df_row, "unicef_region")
        status = getattr(df_row, "status")

        # Populate the SDMX structure for hazards
        for col, indicator in hazards_variable.items():
            sdmx_hazards_structure["DATAFLOW"].append(f"UNICEF{draft_text}:HAZARD_INTENSITY(1.0)")
            sdmx_hazards_structure["REF_AREA"].append(adm2_ucode)
            sdmx_hazards_structure["INDICATOR"].append(f"HAZARD_{hazard_indicator}")
            sdmx_hazards_structure["MEASURE"].append(indicator["measure"])
            sdmx_hazards_structure["ADMIN_LEVEL"].append(admin_level)
            sdmx_hazards_structure["ISO3_PARENT"].append(iso3)
            sdmx_hazards_structure["TIME_PERIOD"].append(time_period)
            sdmx_hazards_structure["OBS_VALUE"].append(getattr(df_row, col))
            sdmx_hazards_structure["UNIT_MEASURE"].append(indicator["unit"])
            sdmx_hazards_structure["REGION"].append(region)
            sdmx_hazards_structure["STATUS"].append(status)

        # Populate the SDMX structure for exposure
        for col, indicator in exposure_variables.items():
            sdmx_exposure_structure["DATAFLOW"].append(f"UNICEF{draft_text}:HAZARD_EXPOSURE(1.0)")
            sdmx_exposure_structure["REF_AREA"].append(adm2_ucode)
            sdmx_exposure_structure["INDICATOR"].append(f"HAZARD_EXPOSURE_{hazard_indicator}")
            sdmx_exposure_structure["MEASURE"].append(indicator["measure"])

            if "f" in col: sex = "F"
            elif "m" in col: sex = "M"
            else: sex = "_T"
            sdmx_exposure_structure["SEX"].append(sex)

            if "u18" in col: age = "Y0T17"
            else: age = "_T"
            sdmx_exposure_structure["AGE"].append(age)
            
            sdmx_exposure_structure["ADMIN_LEVEL"].append(admin_level)
            sdmx_exposure_structure["ISO3_PARENT"].append(iso3)
            sdmx_exposure_structure["TIME_PERIOD"].append(time_period)
            sdmx_exposure_structure["OBS_VALUE"].append(getattr(df_row, col))
            sdmx_exposure_structure["UNIT_MEASURE"].append(indicator["unit"])
            sdmx_exposure_structure["REGION"].append(region)
            sdmx_exposure_structure["STATUS"].append(status)

        # Populate the SDMX structure for population
        if not population_calculated:
            for col, indicator in population_variables.items():
                sdmx_population_structure["DATAFLOW"].append(f"UNICEF{draft_text}:HAZARD_POPULATION(1.0)")
                sdmx_population_structure["REF_AREA"].append(adm2_ucode) 
                sdmx_population_structure["INDICATOR"].append("HAZARD_POPULATION")
                
                if "f" in col: sex = "F"
                elif "m" in col: sex = "M"
                else: sex = "_T"
                sdmx_population_structure["SEX"].append(sex)

                if "u18" in col: age = "Y0T17"
                else: age = "_T"
                sdmx_population_structure["AGE"].append(age)

                sdmx_population_structure["ADMIN_LEVEL"].append(admin_level)
                sdmx_population_structure["ISO3_PARENT"].append(iso3)
                sdmx_population_structure["TIME_PERIOD"].append(time_period)
                sdmx_population_structure["OBS_VALUE"].append(getattr(df_row, col))
                sdmx_population_structure["UNIT_MEASURE"].append(indicator["unit"])
                sdmx_population_structure["REGION"].append(region)
                sdmx_population_structure["STATUS"].append(status)   

    # Save hazard SDMX for all countries for specific hazard
    sdmx_hazard_df = pd.DataFrame(sdmx_hazards_structure)
    path_excel = f"{sdmx_hazard_folder_path}/excel/HAZARD_SDMX_hazard_{hazard_code}_{prod_date}.xlsx"
    path_csv = f"{sdmx_hazard_folder_path}/CSV/HAZARD_SDMX_hazard_{hazard_code}_{prod_date}.csv"
    sdmx_hazard_df.to_excel(path_excel, index=False)
    sdmx_hazard_df.to_csv(path_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

    # Save exposure SDMX for all countries for specific hazard
    sdmx_exposure_df = pd.DataFrame(sdmx_exposure_structure)
    path_excel = f"{sdmx_exposure_folder_path}/excel/HAZARD_SDMX_exposure_{hazard_code}_{prod_date}.xlsx"
    path_csv = f"{sdmx_exposure_folder_path}/CSV/HAZARD_SDMX_exposure_{hazard_code}_{prod_date}.csv"
    sdmx_exposure_df.to_excel(path_excel, index=False)
    sdmx_exposure_df.to_csv(path_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
    
    # The population SDMX is only done once, as the population data is the same for all hazards. 
    # It is calculated during the first iteration and then skipped for the rest of the hazards.
    if not population_calculated:
        population_calculated = True
        sdmx_population_df = pd.DataFrame(sdmx_population_structure)

        path_excel = f"{sdmx_population_folder_path}/excel/HAZARD_SDMX_{prod_date}.xlsx"
        path_csv = f"{sdmx_population_folder_path}/CSV/HAZARD_SDMX_{prod_date}.csv"
        sdmx_population_df.to_excel(path_excel, index=False)
        sdmx_population_df.to_csv(path_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
